In [ ]:
# CIR benchmark suite: standalone CPU/NumPy Kaggle notebook.
# The single cell contains the numerical kernels, metrics, plotting, and run
# orchestration. It does not clone, write, or import the local repository.

from __future__ import annotations

import csv
import os
import time
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import trapezoid
from scipy.stats import ncx2


RUN_MODE = os.environ.get("CIR_SUITE_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("CIR_SUITE_RUN_MODE must be 'full' or 'smoke'")

WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORKING_ROOT / "cir_benchmark_suite_numpy_outputs"
FIG_DIR = OUT_DIR / "figures"
RES_DIR = OUT_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 120339106
FINAL_TIME = 1.0
SHARED = dict(kappa=2.0, theta=0.02, x0=0.02)
REGIMES = {
    "A": {"sigma": 0.10, "description": "Feller well satisfied"},
    "B": {"sigma": 0.20, "description": "Feller satisfied"},
    "C": {"sigma": 0.282842712474619, "description": "Feller boundary"},
    "D": {"sigma": 0.50, "description": "Feller violating"},
    "E": {"sigma": 0.80, "description": "strong Feller violation"},
}

if RUN_MODE == "smoke":
    STRONG_GRID = dict(n_paths=512, reference_n_steps=512, coarse_n_steps=[8, 16, 32])
    DIST_N_PATHS = 1000
    KLM_DIAG_PATHS = 512
    CEV_N_PATHS = 512
    REGIME_LIST = ["A", "B", "C", "D", "E"]
else:
    STRONG_GRID = dict(
        n_paths=20000,
        reference_n_steps=int(os.environ.get("CIR_SUITE_REFERENCE_N_STEPS", "16384")),
        coarse_n_steps=[8, 16, 32, 64, 128, 256],
    )
    DIST_N_PATHS = 50000
    KLM_DIAG_PATHS = 20000
    CEV_N_PATHS = 20000
    REGIME_LIST = ["A", "B", "C", "D", "E"]


def make_rng(seed: int) -> np.random.Generator:
    if seed < 0:
        raise ValueError("seed must be non-negative")
    return np.random.default_rng(seed)


def brownian_increments(rng: np.random.Generator, n_paths: int, n_steps: int, dt: float) -> np.ndarray:
    return np.sqrt(dt) * rng.standard_normal((n_paths, n_steps))


def aggregate_brownian_increments(dW_fine: np.ndarray, factor: int) -> np.ndarray:
    n_paths, n_fine = dW_fine.shape
    if n_fine % factor != 0:
        raise ValueError("fine grid length must be divisible by factor")
    return dW_fine.reshape(n_paths, n_fine // factor, factor).sum(axis=2)


def cir_delta(kappa: float, theta: float, sigma: float) -> float:
    return 4.0 * kappa * theta / sigma**2


def kl_alpha(kappa: float, theta: float, sigma: float) -> float:
    return (4.0 * kappa * theta - sigma**2) / 8.0


def kl_coefficients(kappa: float, theta: float, sigma: float) -> tuple[float, float, float]:
    return kl_alpha(kappa, theta, sigma), -0.5 * kappa, 0.5 * sigma


def cir_ncx2_params(x0: float, kappa: float, theta: float, sigma: float, dt: float):
    df = 4.0 * kappa * theta / sigma**2
    c = 4.0 * kappa / (sigma**2 * (1.0 - np.exp(-kappa * dt)))
    nc = c * x0 * np.exp(-kappa * dt)
    return c, df, nc


def exact_terminal_samples(x0, kappa, theta, sigma, T, n_paths, rng):
    c, df, nc = cir_ncx2_params(x0, kappa, theta, sigma, T)
    return rng.noncentral_chisquare(df, nc, size=n_paths) / c


def exact_terminal_cdf(x, x0, kappa, theta, sigma, T):
    c, df, nc = cir_ncx2_params(x0, kappa, theta, sigma, T)
    return ncx2.cdf(c * np.asarray(x), df, nc)


def exact_terminal_quantile(q, x0, kappa, theta, sigma, T):
    c, df, nc = cir_ncx2_params(x0, kappa, theta, sigma, T)
    return ncx2.ppf(np.asarray(q), df, nc) / c


def exact_cir_mean(x0, kappa, theta, T):
    return float(theta + (x0 - theta) * np.exp(-kappa * T))


def ks_statistic_vs_exact(samples, x0, kappa, theta, sigma, T):
    x = np.sort(np.asarray(samples, dtype=float))
    n = x.size
    cdf = exact_terminal_cdf(x, x0, kappa, theta, sigma, T)
    upper = np.arange(1, n + 1) / n - cdf
    lower = cdf - np.arange(0, n) / n
    return float(np.max(np.maximum(upper, lower)))


def wasserstein1_vs_exact(samples, x0, kappa, theta, sigma, T, n_grid=2048, tail_quantile=0.999):
    x = np.asarray(samples, dtype=float)
    upper = max(
        float(exact_terminal_quantile(tail_quantile, x0, kappa, theta, sigma, T)),
        float(np.max(x)),
    )
    grid = np.linspace(0.0, upper, n_grid)
    exact_cdf = exact_terminal_cdf(grid, x0, kappa, theta, sigma, T)
    empirical_cdf = np.searchsorted(np.sort(x), grid, side="right") / x.size
    return float(trapezoid(np.abs(empirical_cdf - exact_cdf), grid))


def fit_loglog_order(step_sizes, errors):
    h = np.asarray(step_sizes, dtype=float)
    e = np.maximum(np.asarray(errors, dtype=float), 1e-300)
    return float(np.polyfit(np.log(h), np.log(e), 1)[0])


def hh_milstein_step(x, kappa, theta, sigma, dt, dW):
    floor = 0.25 * sigma**2 * dt
    r1 = np.maximum(
        0.5 * sigma * np.sqrt(dt),
        np.sqrt(np.maximum(floor, x)) + 0.5 * sigma * dW,
    )
    x_hat = r1**2 + dt * (kappa * (theta - x) - 0.25 * sigma**2)
    return np.maximum(x_hat, 0.0)


def hh_milstein_terminal_from_dW(X0, kappa, theta, sigma, dt, dW):
    x = np.full(dW.shape[0], X0, dtype=float)
    for n in range(dW.shape[1]):
        x = hh_milstein_step(x, kappa, theta, sigma, dt, dW[:, n])
    return x


def fte_terminal_from_dW(X0, kappa, theta, sigma, dt, dW):
    x_aux = np.full(dW.shape[0], X0, dtype=float)
    for n in range(dW.shape[1]):
        x_pos = np.maximum(x_aux, 0.0)
        x_aux = x_aux + kappa * (theta - x_pos) * dt + sigma * np.sqrt(x_pos) * dW[:, n]
    return np.maximum(x_aux, 0.0)


def projected_euler_terminal_from_dW(X0, kappa, theta, sigma, dt, dW, y_floor=None):
    alpha = kl_alpha(kappa, theta, sigma)
    if y_floor is None:
        y_floor = 0.5 * sigma * np.sqrt(dt)
    y = np.full(dW.shape[0], max(np.sqrt(X0), y_floor), dtype=float)
    for n in range(dW.shape[1]):
        y_safe = np.maximum(y, y_floor)
        y_hat = y_safe + (alpha / y_safe - 0.5 * kappa * y_safe) * dt + 0.5 * sigma * dW[:, n]
        y = np.maximum(y_hat, y_floor)
    return y**2


def kl_uniform_terminal_from_dW(X0, kappa, theta, sigma, dt, dW):
    alpha = kl_alpha(kappa, theta, sigma)
    if alpha < 0.0:
        raise ValueError("KL uniform splitting requires alpha >= 0")
    decay = np.exp(-kappa * dt)
    x = np.full(dW.shape[0], X0, dtype=float)
    for n in range(dW.shape[1]):
        x = decay * (np.sqrt(x + 2.0 * alpha * dt) + 0.5 * sigma * dW[:, n]) ** 2
    return x


def soft_zero_threshold(kappa, theta, dt_max, rho=2.0):
    return theta * (1.0 - np.exp(-kappa * dt_max)) / rho


def kl_adaptive_terminal_from_fine_dW(X0, kappa, theta, sigma, T, dt_max, dW_fine, rho=2.0, safety=0.95):
    alpha = kl_alpha(kappa, theta, sigma)
    X_zero = soft_zero_threshold(kappa, theta, dt_max, rho)
    n_paths, n_fine = dW_fine.shape
    dt_fine = T / n_fine
    W = np.concatenate([np.zeros((n_paths, 1)), np.cumsum(dW_fine, axis=1)], axis=1)
    rows = np.arange(n_paths)
    x = np.full(n_paths, X0, dtype=float)
    pos = np.zeros(n_paths, dtype=np.int64)
    n_steps_total = 0
    n_soft_zero = 0
    while np.any(pos < n_fine):
        idx = np.flatnonzero(pos < n_fine)
        x_a = x[idx]
        remaining_steps = n_fine - pos[idx]
        remaining_time = remaining_steps * dt_fine
        in_soft_zero = x_a < X_zero
        in_splitting = ~in_soft_zero
        h_continuous = np.zeros_like(x_a)
        if np.any(in_soft_zero):
            xs = x_a[in_soft_zero]
            dt_sz = -np.log((X_zero - theta) / (xs - theta)) / kappa
            h_continuous[in_soft_zero] = np.minimum(dt_sz, remaining_time[in_soft_zero])
        if np.any(in_splitting):
            if alpha < 0.0:
                dt_adaptive = safety * x_a[in_splitting] / (2.0 * abs(alpha))
                h_continuous[in_splitting] = np.minimum(
                    np.minimum(dt_adaptive, dt_max), remaining_time[in_splitting]
                )
            else:
                h_continuous[in_splitting] = np.minimum(dt_max, remaining_time[in_splitting])
        m = np.floor(h_continuous / dt_fine).astype(np.int64)
        m = np.maximum(m, 1)
        m = np.minimum(m, remaining_steps)
        h = m * dt_fine
        x_next = x_a.copy()
        if np.any(in_soft_zero):
            h_soft = h[in_soft_zero]
            decay = np.exp(-kappa * h_soft)
            x_next[in_soft_zero] = decay * x_a[in_soft_zero] + theta * (1.0 - decay)
            n_soft_zero += int(np.count_nonzero(in_soft_zero))
        if np.any(in_splitting):
            h_split = h[in_splitting]
            split_idx = idx[in_splitting]
            dW = W[split_idx, pos[split_idx] + m[in_splitting]] - W[
                split_idx, pos[split_idx]
            ]
            inside_sqrt = np.maximum(x_a[in_splitting] + 2.0 * alpha * h_split, 0.0)
            x_next[in_splitting] = np.exp(-kappa * h_split) * (
                np.sqrt(inside_sqrt) + 0.5 * sigma * dW
            ) ** 2
        x[idx] = x_next
        pos[idx] = pos[idx] + m
        n_steps_total += idx.size
    return x, {
        "n_steps_total": n_steps_total,
        "mean_steps_per_path": n_steps_total / n_paths,
        "soft_zero_fraction": n_soft_zero / max(n_steps_total, 1),
    }


def if_step(y, alpha, beta, gamma, dt, dW):
    a = 1.0 - beta * dt
    b = -(y + gamma * dW)
    c = -alpha * dt
    return (-b + np.sqrt(b * b - 4.0 * a * c)) / (2.0 * a)


def default_backstop_kind(alpha):
    return "implicit" if alpha > 0.0 else "projected"


def backstop_map(y, h, dW, alpha, beta, gamma, kind):
    if kind == "implicit":
        if alpha <= 0.0:
            raise ValueError("implicit backstop requires alpha > 0")
        return if_step(y, alpha, beta, gamma, h, dW)
    y_floor = gamma * np.sqrt(h)
    y_hat = y + (alpha / y + beta * y) * h + gamma * dW
    return np.maximum(y_hat, y_floor)


def klm_backstop_terminal(X0, kappa, theta, sigma, T, h_max, n_paths, rng, rho=64.0):
    alpha, beta, gamma = kl_coefficients(kappa, theta, sigma)
    kind = default_backstop_kind(alpha)
    h_min = h_max / rho
    y = np.full(n_paths, np.sqrt(X0), dtype=float)
    t = np.zeros(n_paths, dtype=float)
    n_steps_total = 0
    n_backstop_min = 0
    n_backstop_neg = 0
    while np.any(t < T - 1e-14):
        idx = np.flatnonzero(t < T - 1e-14)
        y_a = y[idx]
        h_prop = h_max * np.minimum(1.0, np.abs(y_a))
        min_triggered = h_prop < h_min
        h = np.where(min_triggered, h_min, h_prop)
        h = np.minimum(h, T - t[idx])
        dW = np.sqrt(h) * rng.standard_normal(idx.size)
        explicit_candidate = y_a + (alpha / y_a + beta * y_a) * h + gamma * dW
        use_backstop = min_triggered | (explicit_candidate <= 0.0)
        y_next = np.where(
            use_backstop,
            backstop_map(y_a, h, dW, alpha, beta, gamma, kind),
            explicit_candidate,
        )
        y[idx] = y_next
        t[idx] += h
        n_steps_total += idx.size
        n_backstop_min += int(np.count_nonzero(min_triggered))
        n_backstop_neg += int(np.count_nonzero((~min_triggered) & (explicit_candidate <= 0.0)))
    stats = {
        "n_steps_total": n_steps_total,
        "n_backstop_min": n_backstop_min,
        "n_backstop_neg": n_backstop_neg,
        "backstop_fraction": (n_backstop_min + n_backstop_neg) / max(n_steps_total, 1),
        "backstop_kind": kind,
    }
    return y**2, stats


def klm_backstop_terminal_from_fine_dW(X0, kappa, theta, sigma, T, h_max, dW_fine, rho=64.0):
    alpha, beta, gamma = kl_coefficients(kappa, theta, sigma)
    kind = default_backstop_kind(alpha)
    n_paths, n_fine = dW_fine.shape
    dt_fine = T / n_fine
    h_min = h_max / rho
    m_min = max(int(round(h_min / dt_fine)), 1)
    m_max = max(int(np.floor(h_max / dt_fine)), 1)
    W = np.concatenate([np.zeros((n_paths, 1)), np.cumsum(dW_fine, axis=1)], axis=1)
    y = np.full(n_paths, np.sqrt(X0), dtype=float)
    pos = np.zeros(n_paths, dtype=np.int64)
    n_steps_total = 0
    n_backstop_min = 0
    n_backstop_neg = 0
    rows = np.arange(n_paths)
    while np.any(pos < n_fine):
        idx = np.flatnonzero(pos < n_fine)
        y_a = y[idx]
        h_prop = h_max * np.minimum(1.0, np.abs(y_a))
        m = np.floor(h_prop / dt_fine).astype(np.int64)
        min_triggered = m < m_min
        m = np.where(min_triggered, m_min, np.minimum(m, m_max))
        m = np.minimum(m, n_fine - pos[idx])
        h = m * dt_fine
        dW = W[rows[idx], pos[idx] + m] - W[rows[idx], pos[idx]]
        explicit_candidate = y_a + (alpha / y_a + beta * y_a) * h + gamma * dW
        use_backstop = min_triggered | (explicit_candidate <= 0.0)
        y_next = np.where(
            use_backstop,
            backstop_map(y_a, h, dW, alpha, beta, gamma, kind),
            explicit_candidate,
        )
        y[idx] = y_next
        pos[idx] += m
        n_steps_total += idx.size
        n_backstop_min += int(np.count_nonzero(min_triggered))
        n_backstop_neg += int(np.count_nonzero((~min_triggered) & (explicit_candidate <= 0.0)))
    stats = {
        "n_steps_total": n_steps_total,
        "n_backstop_min": n_backstop_min,
        "n_backstop_neg": n_backstop_neg,
        "backstop_fraction": (n_backstop_min + n_backstop_neg) / max(n_steps_total, 1),
        "backstop_kind": kind,
        "m_min_fine_steps": m_min,
    }
    return y**2, stats


def terminal_samples(scheme, params, n_steps, n_paths, rng):
    dt = params["T"] / n_steps
    if scheme == "Exact":
        return exact_terminal_samples(params["x0"], params["kappa"], params["theta"], params["sigma"], params["T"], n_paths, rng)
    if scheme == "KLM":
        terminal, _ = klm_backstop_terminal(
            X0=params["x0"], kappa=params["kappa"], theta=params["theta"],
            sigma=params["sigma"], T=params["T"], h_max=params["T"] / n_steps,
            n_paths=n_paths, rng=rng,
        )
        return terminal
    dW = brownian_increments(rng, n_paths, n_steps, dt)
    if scheme == "FTE":
        return fte_terminal_from_dW(params["x0"], params["kappa"], params["theta"], params["sigma"], dt, dW)
    if scheme == "HH":
        return hh_milstein_terminal_from_dW(params["x0"], params["kappa"], params["theta"], params["sigma"], dt, dW)
    if scheme == "ProjEuler":
        return projected_euler_terminal_from_dW(params["x0"], params["kappa"], params["theta"], params["sigma"], dt, dW)
    raise ValueError(scheme)


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def run_strong_error_suite() -> list[dict]:
    rows = []
    schemes = ["FTE", "ProjEuler", "KL", "KLM"]
    styles = {
        "FTE": dict(color="tab:blue", marker="o"),
        "ProjEuler": dict(color="tab:green", marker="^"),
        "KL": dict(color="tab:red", marker="d"),
        "KLM": dict(color="tab:purple", marker="v"),
    }
    for regime_name in REGIME_LIST:
        sigma = REGIMES[regime_name]["sigma"]
        params = dict(**SHARED, sigma=sigma, T=FINAL_TIME)
        rng = make_rng(MASTER_SEED)
        dt_fine = FINAL_TIME / STRONG_GRID["reference_n_steps"]
        dW_fine = brownian_increments(rng, STRONG_GRID["n_paths"], STRONG_GRID["reference_n_steps"], dt_fine)
        reference = hh_milstein_terminal_from_dW(params["x0"], params["kappa"], params["theta"], sigma, dt_fine, dW_fine)
        for scheme in schemes:
            for n_steps in STRONG_GRID["coarse_n_steps"]:
                factor = STRONG_GRID["reference_n_steps"] // n_steps
                dW = aggregate_brownian_increments(dW_fine, factor)
                dt = FINAL_TIME / n_steps
                start = time.perf_counter()
                if scheme == "FTE":
                    terminal = fte_terminal_from_dW(params["x0"], params["kappa"], params["theta"], sigma, dt, dW)
                    mean_steps = n_steps
                    backstop_fraction = 0.0
                    scheme_variant = "fixed"
                elif scheme == "ProjEuler":
                    terminal = projected_euler_terminal_from_dW(params["x0"], params["kappa"], params["theta"], sigma, dt, dW)
                    mean_steps = n_steps
                    backstop_fraction = 0.0
                    scheme_variant = "fixed"
                elif scheme == "KL":
                    if kl_alpha(params["kappa"], params["theta"], sigma) < 0.0:
                        terminal, stats = kl_adaptive_terminal_from_fine_dW(
                            params["x0"], params["kappa"], params["theta"], sigma,
                            FINAL_TIME, dt, dW_fine,
                        )
                        mean_steps = stats["mean_steps_per_path"]
                        backstop_fraction = stats["soft_zero_fraction"]
                        scheme_variant = "adaptive-soft-zero"
                    else:
                        terminal = kl_uniform_terminal_from_dW(params["x0"], params["kappa"], params["theta"], sigma, dt, dW)
                        mean_steps = n_steps
                        backstop_fraction = 0.0
                        scheme_variant = "uniform"
                else:
                    terminal, stats = klm_backstop_terminal_from_fine_dW(
                        params["x0"], params["kappa"], params["theta"], sigma,
                        FINAL_TIME, dt, dW_fine,
                    )
                    mean_steps = stats["n_steps_total"] / STRONG_GRID["n_paths"]
                    backstop_fraction = stats["backstop_fraction"]
                    scheme_variant = stats["backstop_kind"]
                runtime = time.perf_counter() - start
                diff = terminal - reference
                rows.append({
                    "regime": regime_name,
                    "scheme": scheme,
                    "scheme_variant": scheme_variant,
                    "reference": "HH",
                    "dt": dt,
                    "l1": float(np.mean(np.abs(diff))),
                    "l2": float(np.sqrt(np.mean(diff**2))),
                    "runtime_s": runtime,
                    "mean_steps_per_path": mean_steps,
                    "backstop_fraction": backstop_fraction,
                })
        regime_rows = [r for r in rows if r["regime"] == regime_name]
        write_csv(RES_DIR / f"strong_error_regime_{regime_name}.csv", regime_rows)
        fig, ax = plt.subplots(figsize=(6.0, 4.4))
        for scheme in schemes:
            sr = [r for r in regime_rows if r["scheme"] == scheme]
            if not sr:
                continue
            h = np.array([r["dt"] for r in sr])
            e = np.array([r["l2"] for r in sr])
            order = fit_loglog_order(h, e) if len(h) >= 2 else np.nan
            ax.loglog(h, e, label=f"{scheme} ({order:.2f})", **styles[scheme])
        ax.set_xlabel("step size h")
        ax.set_ylabel("strong L2 error at T")
        ax.set_title(f"Regime {regime_name}: schemes vs HH reference")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"strong_error_regime_{regime_name}.pdf")
        plt.close(fig)
    return rows


def run_klm_diagnostic() -> list[dict]:
    rows = []
    h_levels = [2.0**(-k) for k in range(3, 10)]
    for regime_name in REGIME_LIST:
        sigma = REGIMES[regime_name]["sigma"]
        params = dict(**SHARED, sigma=sigma, T=FINAL_TIME)
        exact_mean = exact_cir_mean(params["x0"], params["kappa"], params["theta"], FINAL_TIME)
        for h_max in h_levels:
            terminal, stats = klm_backstop_terminal(
                params["x0"], params["kappa"], params["theta"], sigma,
                FINAL_TIME, h_max, KLM_DIAG_PATHS, make_rng(MASTER_SEED),
            )
            rows.append({
                "regime": regime_name,
                "sigma": sigma,
                "delta": cir_delta(params["kappa"], params["theta"], sigma),
                "alpha": kl_alpha(params["kappa"], params["theta"], sigma),
                "h_max": h_max,
                "n_paths": KLM_DIAG_PATHS,
                "sample_mean": float(np.mean(terminal)),
                "exact_mean": exact_mean,
                "mean_error": float(np.mean(terminal) - exact_mean),
                **stats,
            })
    write_csv(RES_DIR / "klm_backstop_diagnostic.csv", rows)
    fig, ax = plt.subplots(figsize=(6.0, 4.4))
    for regime_name in REGIME_LIST:
        sr = [r for r in rows if r["regime"] == regime_name]
        ax.semilogx([r["h_max"] for r in sr], [r["backstop_fraction"] for r in sr], "o-", label=regime_name)
    ax.set_xlabel("h_max")
    ax.set_ylabel("backstop fraction")
    ax.grid(True, alpha=0.3)
    ax.legend(title="regime")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "klm_backstop_diagnostic.pdf")
    plt.close(fig)
    return rows


def run_distributional() -> list[dict]:
    rows = []
    n_grid = [8, 16, 32, 64, 128, 256] if RUN_MODE == "full" else [8, 16, 32]
    for regime_name in [r for r in ["A", "C", "E"] if r in REGIME_LIST]:
        sigma = REGIMES[regime_name]["sigma"]
        params = dict(**SHARED, sigma=sigma, T=FINAL_TIME)
        for n_steps in n_grid:
            for scheme in ["FTE", "HH", "ProjEuler", "KLM", "Exact"]:
                terminal = terminal_samples(
                    scheme, params, n_steps, DIST_N_PATHS, make_rng(MASTER_SEED + n_steps)
                )
                rows.append({
                    "regime": regime_name,
                    "scheme": scheme,
                    "n_steps": n_steps,
                    "n_paths": DIST_N_PATHS,
                    "ks": ks_statistic_vs_exact(terminal, params["x0"], params["kappa"], params["theta"], sigma, FINAL_TIME),
                    "w1": wasserstein1_vs_exact(terminal, params["x0"], params["kappa"], params["theta"], sigma, FINAL_TIME),
                    "terminal_mean": float(np.mean(terminal)),
                })
    write_csv(RES_DIR / "distributional_diagnostics.csv", rows)
    fig, ax = plt.subplots(figsize=(6.0, 4.4))
    for scheme in ["FTE", "HH", "ProjEuler", "KLM", "Exact"]:
        sr = [r for r in rows if r["scheme"] == scheme and r["regime"] == "C"]
        if sr:
            ax.loglog([r["n_steps"] for r in sr], [r["ks"] for r in sr], "o-", label=scheme)
    ax.set_xlabel("number of steps")
    ax.set_ylabel("KS distance, regime C")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "distributional_diagnostics.pdf")
    plt.close(fig)
    return rows


def cev_exact_mean(F0, kappa, theta, T):
    return float(theta + (F0 - theta) * np.exp(-kappa * T))


def y_from_f(f, beta):
    return f ** (1.0 - beta) / (1.0 - beta)


def f_from_y(y, beta):
    return ((1.0 - beta) * y) ** (1.0 / (1.0 - beta))


def cev_lamperti_drift(y, kappa, theta, sigma, beta):
    one_minus_beta = 1.0 - beta
    return (
        kappa * theta * (one_minus_beta * y) ** (-beta / one_minus_beta)
        - kappa * one_minus_beta * y
        - beta * sigma**2 / (2.0 * one_minus_beta * y)
    )


def cev_projected_terminal_from_dW(F0, kappa, theta, sigma, beta, dt, dW, y_floor=None):
    if y_floor is None:
        y_floor = sigma * np.sqrt(dt)
    y = np.full(dW.shape[0], max(y_from_f(F0, beta), y_floor), dtype=float)
    for n in range(dW.shape[1]):
        y_safe = np.maximum(y, y_floor)
        y_hat = y_safe + cev_lamperti_drift(y_safe, kappa, theta, sigma, beta) * dt + sigma * dW[:, n]
        y = np.maximum(y_hat, y_floor)
    return f_from_y(y, beta)


def run_cev_experiment() -> list[dict]:
    rows = []
    sigma = REGIMES["B"]["sigma"]
    coarse = [8, 16, 32, 64, 128, 256] if RUN_MODE == "full" else [8, 16, 32]
    reference_n_steps = 4096 if RUN_MODE == "full" else 512
    dt_fine = FINAL_TIME / reference_n_steps
    for beta in [0.5, 0.75]:
        dW_fine = brownian_increments(make_rng(MASTER_SEED), CEV_N_PATHS, reference_n_steps, dt_fine)
        reference = cev_projected_terminal_from_dW(SHARED["x0"], SHARED["kappa"], SHARED["theta"], sigma, beta, dt_fine, dW_fine)
        for n_steps in coarse:
            factor = reference_n_steps // n_steps
            dW = aggregate_brownian_increments(dW_fine, factor)
            dt = FINAL_TIME / n_steps
            terminal = cev_projected_terminal_from_dW(SHARED["x0"], SHARED["kappa"], SHARED["theta"], sigma, beta, dt, dW)
            diff = terminal - reference
            rows.append({
                "beta": beta,
                "n_steps": n_steps,
                "dt": dt,
                "n_paths": CEV_N_PATHS,
                "l2": float(np.sqrt(np.mean(diff**2))),
                "sample_mean": float(np.mean(terminal)),
                "exact_mean": cev_exact_mean(SHARED["x0"], SHARED["kappa"], SHARED["theta"], FINAL_TIME),
            })
    write_csv(RES_DIR / "cev_convergence.csv", rows)
    fig, ax = plt.subplots(figsize=(6.0, 4.4))
    for beta in [0.5, 0.75]:
        sr = [r for r in rows if r["beta"] == beta]
        ax.loglog([r["dt"] for r in sr], [r["l2"] for r in sr], "o-", label=f"beta={beta:g}")
    ax.set_xlabel("step size h")
    ax.set_ylabel("L2 self-reference error")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "cev_convergence.pdf")
    plt.close(fig)
    return rows


def run_order_summary(strong_rows: list[dict]) -> list[dict]:
    rows = []
    for regime_name in sorted({r["regime"] for r in strong_rows}):
        sigma = REGIMES[regime_name]["sigma"]
        for scheme in sorted({r["scheme"] for r in strong_rows if r["regime"] == regime_name}):
            sr = [r for r in strong_rows if r["regime"] == regime_name and r["scheme"] == scheme]
            if len(sr) < 2:
                continue
            rows.append({
                "regime": regime_name,
                "scheme": scheme,
                "sigma": sigma,
                "delta": cir_delta(SHARED["kappa"], SHARED["theta"], sigma),
                "fitted_l2_order": fit_loglog_order([r["dt"] for r in sr], [r["l2"] for r in sr]),
            })
    write_csv(RES_DIR / "fig_order_vs_delta_summary.csv", rows)
    return rows


def archive_outputs() -> None:
    import shutil

    archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    print(f"Archived outputs to {archive}")


def main() -> None:
    print(f"Standalone CPU/NumPy CIR benchmark suite, RUN_MODE={RUN_MODE}")
    print(f"Output directory: {OUT_DIR}")
    strong_rows = run_strong_error_suite()
    run_order_summary(strong_rows)
    run_klm_diagnostic()
    run_distributional()
    run_cev_experiment()
    archive_outputs()
    print("Done.")


main()
